In [1]:
!pip install gradio


In [ ]:
##Muscle Atropy detection

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
import gradio as gr

# ---------------------------
# Load Training Data
# ---------------------------
df = pd.read_csv('/36.csv')

identifier_cols = ['GSM_ID', 'Subject_ID', 'Group', 'Timepoint']
target_col = 'Target'
feature_cols = [col for col in df.columns if col not in identifier_cols + [target_col]]

X = df[feature_cols]
y = df[target_col]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train base models
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
xgb = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric="logloss")
svm = SVC(probability=True, random_state=42, class_weight="balanced", kernel="rbf", C=1.0)

rf.fit(X_scaled, y)
xgb.fit(X_scaled, y)
svm.fit(X_scaled, y)

# Train Ensemble
ensemble = VotingClassifier(
    estimators=[("rf", rf), ("xgb", xgb), ("svm", svm)],
    voting="soft"
)
ensemble.fit(X_scaled, y)

# Dictionary of models
models = {
    "Random Forest": rf,
    "XGBoost": xgb,
    "SVM (RBF Kernel)": svm,
    "Ensemble Voting Classifier": ensemble
}

# ---------------------------
# Prediction Function
# ---------------------------
def predict(file, model_choice):
    test_df = pd.read_csv(file.name)   # user uploads test CSV
    X_test = test_df[feature_cols]
    X_test_scaled = scaler.transform(X_test)

    model = models[model_choice]
    preds = model.predict(X_test_scaled)

    test_df["Prediction"] = np.where(preds == 0, "Normal", "Atrophied")
    return test_df[["Subject_ID", "Prediction"]]

# ---------------------------
# Gradio Interface
# ---------------------------
iface = gr.Interface(
    fn=predict,
    inputs=[
        gr.File(label="Upload Test CSV"),
        gr.Dropdown(choices=list(models.keys()), label="Choose Model")
    ],
    outputs="dataframe",
    title="Muscle Atrophy Detection",
    description="Upload a test CSV file to check if subjects are Normal or Atrophied."
)

iface.launch(share=True)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:46:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:46:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://412e83277e4bebd2de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
